# MODEL & TRAINING FLOW (GPT‑1 style, decoder-only)

1) DATA & TOKENIZATION
   - We build a tiny vocabulary from raw text.
   - Every sentence is turned into a sequence of token IDs.
   - Special tokens:
       <pad> = padding
       <unk> = unknown token
       <cls> = classification token (for sentiment task)

2) PRETRAINING: CAUSAL LANGUAGE MODEL (LM)
   Goal: Learn to predict the NEXT TOKEN (no human labels).

   For each training sentence:
     - Turn text into IDs: [w1, w2, w3, ..., wN]
     - Inputs:  [w1, w2, ..., w(N-1)]
     - Targets: [w2, w3, ..., wN]
   So the model learns:  P(w_t | w_1, ..., w_(t-1))

   Model architecture (GPT‑1 style):
     - Input IDs -> token embedding (tok_emb)
     - Add positional embedding (pos_emb)
     - Pass through several Transformer "blocks":
         * LayerNorm
         * Masked multi-head self-attention (decoder-only, no cross-attention)
         * Residual connection
         * LayerNorm
         * Feed-forward network (FFN) with GELU
         * Residual connection
     - Final LayerNorm (ln_f)
     - Linear layer to vocab size (lm_head) to get logits for each token

   Causal Mask:
     - Each token can only attend to itself and PAST tokens.
     - Future tokens are masked out so the model really "predicts next token".

   Loss:
     - Cross-entropy between predicted token distribution and true next token.
     - No manual labels: we use the text itself ("self-supervised").

3) DEMO AFTER PRETRAINING: GENERATION
   - Give a prompt like "this movie is".
   - Model predicts next token step by step (greedy decoding).
   - Shows students the model has learned patterns from raw text only.

4) SAVE PRETRAINED WEIGHTS
   - We store:
       embeddings, LayerNorm parameters, attention weights, FFN weights, lm_head.
   - This is the pretrained "GPT‑like" base.

5) RELOAD & FREEZE GPT FOR DOWNSTREAM TASK
   - Reload all weights into the same structure.
   - Set requires_grad = False for ALL GPT parameters.
   - GPT now acts as a FROZEN FEATURE EXTRACTOR.

6) SENTIMENT CLASSIFICATION (DOWNSTREAM TASK)
   Goal: Use frozen GPT to classify sentiment (POSITIVE / NEGATIVE)
         with only a few labeled examples.

   Data:
     - Short sentences with labels (1 = positive, 0 = negative).
     - We add a <cls> token at the start:
         "<cls> this movie is great"
     - Pad to fixed length.

   Forward pass for classification:
     1) Pass input IDs through frozen GPT (same as LM, but no LM loss here).
        - Get last hidden states: [B, T, D]
     2) Take hidden state at <cls> position: [B, D]
        - This is our sentence representation.
     3) Apply a small trainable classifier:
           logits = Linear([B, D] -> [B, 2])
     4) Compute cross-entropy loss with sentiment labels.

   Training:
     - Only classifier parameters are updated (GPT weights stay frozen).
     - After training, the classifier learns to map GPT features -> sentiment.

7) TESTING
   - Give new sentences (never seen before) to GPT + classifier.
   - GPT produces features, classifier outputs POSITIVE / NEGATIVE.
   - This shows how a decoder-only LM can be reused for NLU tasks.

In [1]:
#pip install torch tqdm
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [2]:
# ============================================================
# STEP 1: Toy data for pretraining (causal LM) + sentiment
# ============================================================
pretrain_texts = [
    "this movie is great",
    "this movie is terrible",
    "i love this film",
    "i hate this film",
    "the food was delicious",
    "the food was awful",
    "today is a good day",
    "today is a bad day",
    "i really enjoyed this movie",
    "i really disliked this movie",
]
# Few-shot sentiment (1=positive, 0=negative)
sentiment_texts = [
    "this movie is great",
    "i love this film",
    "the food was delicious",
    "today is a good day",
    "this movie is terrible",
    "i hate this film",
    "the food was awful",
    "today is a bad day",
]
sentiment_labels = [1, 1, 1, 1, 0, 0, 0, 0]

In [3]:
# ============================================================
# STEP 2: Simple tokenizer (no class)
# ============================================================
from collections import Counter
# build vocab from pretrain_texts
counter = Counter()
for t in pretrain_texts:
    counter.update(t.lower().split())
PAD = "<pad>"
UNK = "<unk>"
CLS = "<cls>"
vocab = [PAD, UNK, CLS]
for w, c in counter.items():
    if w not in vocab:
        vocab.append(w)
stoi = {w: i for i, w in enumerate(vocab)}
itos = {i: w for w, i in stoi.items()}
vocab_size = len(vocab)
pad_id = stoi[PAD]
unk_id = stoi[UNK]
cls_id = stoi[CLS]
print("Vocab size:", vocab_size)
print("Vocab:", vocab)
def encode(text, add_cls=False):
    tokens = text.lower().split()
    ids = []
    if add_cls:
        ids.append(cls_id)
    for tok in tokens:
        ids.append(stoi.get(tok, unk_id))
    return ids
def decode(ids):
    return " ".join(itos.get(i, UNK) for i in ids)


Vocab size: 25
Vocab: ['<pad>', '<unk>', '<cls>', 'this', 'movie', 'is', 'great', 'terrible', 'i', 'love', 'film', 'hate', 'the', 'food', 'was', 'delicious', 'awful', 'today', 'a', 'good', 'day', 'bad', 'really', 'enjoyed', 'disliked']


In [5]:
# ============================================================
# STEP 3: Build pretraining (LM) tensors with causal targets
# ============================================================
max_len = 16  # small for demo
lm_inputs = []
lm_targets = []
for text in pretrain_texts:
    ids = encode(text, add_cls=False)
    # create (input, target) pairs: predict next token
    if len(ids) < 2:
        continue
    # chunk if longer than max_len
    for i in range(0, len(ids) - 1, max_len):
        chunk = ids[i:i + max_len]
        if len(chunk) < 2:
            continue
        inp = chunk[:-1]
        tgt = chunk[1:]
        # pad
        T = len(inp)
        pad_len = max_len - T
        inp = inp + [pad_id] * pad_len
        tgt = tgt + [-100] * pad_len    # ignore_index for loss
        lm_inputs.append(inp)
        lm_targets.append(tgt)
lm_inputs = torch.tensor(lm_inputs, dtype=torch.long)
lm_targets = torch.tensor(lm_targets, dtype=torch.long)
print("LM dataset size:", lm_inputs.shape)
lm_dataset = TensorDataset(lm_inputs, lm_targets)
lm_loader = DataLoader(lm_dataset, batch_size=8, shuffle=True)


LM dataset size: torch.Size([10, 16])


In [6]:
# ============================================================
# STEP 4: Build sentiment tensors (few-shot)
# ============================================================
sent_inputs = []
sent_labels_t = []
for text, label in zip(sentiment_texts, sentiment_labels):
    ids = encode(text, add_cls=True)  # prepend <cls>
    if len(ids) > max_len:
        ids = ids[:max_len]
    pad_len = max_len - len(ids)
    ids = ids + [pad_id] * pad_len
    sent_inputs.append(ids)
    sent_labels_t.append(label)
sent_inputs = torch.tensor(sent_inputs, dtype=torch.long)
sent_labels_t = torch.tensor(sent_labels_t, dtype=torch.long)
print("Sentiment dataset size:", sent_inputs.shape)
sent_dataset = TensorDataset(sent_inputs, sent_labels_t)
sent_loader = DataLoader(sent_dataset, batch_size=4, shuffle=True)


Sentiment dataset size: torch.Size([8, 16])


In [7]:
# ============================================================
# STEP 5: Define GPT-1 style model pieces (no classes)
# ============================================================
# Hyperparameters
d_model = 128
n_layers = 2
n_heads = 4
d_ff = 256
# Embeddings
tok_emb = nn.Embedding(vocab_size, d_model).to(device)
pos_emb = nn.Embedding(max_len, d_model).to(device)
# Initialize Transformer block parameters as lists (for each layer)
ln1_weights = []
ln1_biases = []
attn_W_q = []
attn_W_k = []
attn_W_v = []
attn_W_o = []
ln2_weights = []
ln2_biases = []
ffn_W1 = []
ffn_b1 = []
ffn_W2 = []
ffn_b2 = []
for layer_idx in range(n_layers):
    # LayerNorm 1
    ln1 = nn.LayerNorm(d_model).to(device)
    ln1_weights.append(ln1.weight)
    ln1_biases.append(ln1.bias)
    # Multi-head self-attention parameters
    W_q = nn.Linear(d_model, d_model, bias=True).to(device)
    W_k = nn.Linear(d_model, d_model, bias=True).to(device)
    W_v = nn.Linear(d_model, d_model, bias=True).to(device)
    W_o = nn.Linear(d_model, d_model, bias=True).to(device)
    attn_W_q.append(W_q)
    attn_W_k.append(W_k)
    attn_W_v.append(W_v)
    attn_W_o.append(W_o)
    # LayerNorm 2
    ln2 = nn.LayerNorm(d_model).to(device)
    ln2_weights.append(ln2.weight)
    ln2_biases.append(ln2.bias)
    # FFN
    linear1 = nn.Linear(d_model, d_ff).to(device)
    linear2 = nn.Linear(d_ff, d_model).to(device)
    ffn_W1.append(linear1.weight)
    ffn_b1.append(linear1.bias)
    ffn_W2.append(linear2.weight)
    ffn_b2.append(linear2.bias)
# Final layer norm and LM head
ln_f = nn.LayerNorm(d_model).to(device)
lm_head = nn.Linear(d_model, vocab_size, bias=False).to(device)


In [8]:
# ============================================================
# STEP 6: Helper functions for attention and block forward
# ============================================================
def get_causal_mask(T, device):
    # [T, T] lower triangular
    mask = torch.tril(torch.ones(T, T, device=device))
    return mask  # 1 where allowed, 0 where masked
def multi_head_self_attention(x, layer_idx):
    # x: [B, T, D]
    B, T, D = x.size()
    H = n_heads
    d_head = D // H
    Wq = attn_W_q[layer_idx]
    Wk = attn_W_k[layer_idx]
    Wv = attn_W_v[layer_idx]
    Wo = attn_W_o[layer_idx]
    q = Wq(x)  # [B, T, D]
    k = Wk(x)
    v = Wv(x)
    # reshape for heads: [B, T, D] -> [B, H, T, d_head]
    q = q.view(B, T, H, d_head).transpose(1, 2)  # [B, H, T, d_head]
    k = k.view(B, T, H, d_head).transpose(1, 2)
    v = v.view(B, T, H, d_head).transpose(1, 2)
    scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(d_head)  # [B, H, T, T]
    mask = get_causal_mask(T, x.device).view(1, 1, T, T)
    scores = scores.masked_fill(mask == 0, float('-inf'))
    attn = F.softmax(scores, dim=-1)  # [B, H, T, T]
    out = torch.matmul(attn, v)      # [B, H, T, d_head]
    out = out.transpose(1, 2).contiguous().view(B, T, D)  # [B, T, D]
    out = Wo(out)  # [B, T, D]
    return out
def gpt_block_forward(x, layer_idx):
    # x: [B, T, D]
    # LN1
    w1 = ln1_weights[layer_idx]
    b1 = ln1_biases[layer_idx]
    # manual LayerNorm via module is messy without class, so we use builtin function:
    # but we have to wrap w1,b1 in a fake LN? To keep simple, call F.layer_norm directly.
    x_norm1 = F.layer_norm(x, (d_model,), weight=w1, bias=b1)
    # Self-attention
    attn_out = multi_head_self_attention(x_norm1, layer_idx)
    x = x + attn_out
    # LN2
    w2 = ln2_weights[layer_idx]
    b2 = ln2_biases[layer_idx]
    x_norm2 = F.layer_norm(x, (d_model,), weight=w2, bias=b2)
    # FFN with GELU
    W1 = ffn_W1[layer_idx]
    b1_ = ffn_b1[layer_idx]
    W2 = ffn_W2[layer_idx]
    b2_ = ffn_b2[layer_idx]
    hidden = F.linear(x_norm2, W1, b1_)   # [B, T, d_ff]
    hidden = F.gelu(hidden)
    hidden = F.linear(hidden, W2, b2_)    # [B, T, D]
    x = x + hidden
    return x
def gpt_forward_lm(input_ids, targets=None):
    # input_ids: [B, T]
    B, T = input_ids.size()
    positions = torch.arange(0, T, device=device).unsqueeze(0)  # [1, T]
    x = tok_emb(input_ids.to(device)) + pos_emb(positions)  # [B, T, D]
    for layer_idx in range(n_layers):
        x = gpt_block_forward(x, layer_idx)
    x = ln_f(x)  # [B, T, D]
    logits = lm_head(x)  # [B, T, vocab_size]
    loss = None
    if targets is not None:
        loss = F.cross_entropy(
            logits.view(-1, vocab_size),
            targets.to(device).view(-1),
            ignore_index=-100
        )
    return logits, loss, x


In [9]:
# ============================================================
# STEP 7: Pretrain LM (causal language modeling)
# ============================================================

params = (
    list(tok_emb.parameters())
    + list(pos_emb.parameters())
    + ln1_weights + ln1_biases
    + [p for layer in attn_W_q for p in layer.parameters()]
    + [p for layer in attn_W_k for p in layer.parameters()]
    + [p for layer in attn_W_v for p in layer.parameters()]
    + [p for layer in attn_W_o for p in layer.parameters()]
    + ln2_weights + ln2_biases
    + ffn_W1 + ffn_b1 + ffn_W2 + ffn_b2
    + list(ln_f.parameters())
    + list(lm_head.parameters())
)

optimizer = torch.optim.AdamW(params, lr=3e-4)

epochs = 10
print("=== Pretraining LM ===")
for epoch in range(epochs):
    total_loss = 0.0
    for inp, tgt in tqdm(lm_loader, desc=f"Epoch {epoch+1}/{epochs}"):
        inp = inp.to(device)
        tgt = tgt.to(device)
        optimizer.zero_grad()
        _, loss, _ = gpt_forward_lm(inp, targets=tgt)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1} LM loss: {total_loss/len(lm_loader):.4f}")

# ============================================================
# STEP 7.5: Demo causal LM generation (next-token prediction)
# ============================================================

def generate_next_tokens(prompt_text, max_new_tokens=5):
    # 1) Encode prompt
    ids = encode(prompt_text, add_cls=False)
    if len(ids) == 0:
        return ""
    # 2) Truncate / pad to max_len
    if len(ids) > max_len:
        ids = ids[-max_len:]  # keep last tokens
    # We'll always feed full max_len to the model
    while len(ids) < max_len:
        ids.append(pad_id)
    input_ids = torch.tensor([ids], dtype=torch.long).to(device)  # [1, T]

    # 3) Generate tokens autoregressively
    generated = ids[:]  # copy
    for _ in range(max_new_tokens):
        # For generation, ignore pad label targets
        _, _, hidden_states = gpt_forward_lm(input_ids, targets=None)
        # logits for last position in sequence
        last_hidden = hidden_states[:, -1, :]             # [1, D]
        last_logits = lm_head(last_hidden)                # [1, vocab_size]
        probs = F.softmax(last_logits, dim=-1)            # [1, vocab_size]
        next_id = torch.argmax(probs, dim=-1).item()      # greedy
        # Append next token to generated sequence (before pads)
        generated.append(next_id)
        # Update input_ids by shifting window over last max_len tokens
        last_tokens = generated[-max_len:]
        # pad if needed
        if len(last_tokens) < max_len:
            last_tokens = [pad_id]*(max_len-len(last_tokens)) + last_tokens
        input_ids = torch.tensor([last_tokens], dtype=torch.long).to(device)

    # 4) Decode only non-pad tokens
    decoded_tokens = [tid for tid in generated if tid != pad_id]
    return decode(decoded_tokens)

print("\n=== Demo: Causal LM next-token prediction ===")
demo_prompts = [
    "this movie is",
    "the food was",
    "today is a",
    "i really",
]

for p in demo_prompts:
    out = generate_next_tokens(p, max_new_tokens=5)
    print(f"Prompt: '{p}' -> Generated: '{out}'")

# After this, we save the weights
torch.save({
    "tok_emb": tok_emb.state_dict(),
    "pos_emb": pos_emb.state_dict(),
    "ln_f": ln_f.state_dict(),
    "lm_head": lm_head.state_dict(),
    "ln1_weights": [w.detach().cpu() for w in ln1_weights],
    "ln1_biases": [b.detach().cpu() for b in ln1_biases],
    "ln2_weights": [w.detach().cpu() for w in ln2_weights],
    "ln2_biases": [b.detach().cpu() for b in ln2_biases],
    "attn_W_q": [layer.state_dict() for layer in attn_W_q],
    "attn_W_k": [layer.state_dict() for layer in attn_W_k],
    "attn_W_v": [layer.state_dict() for layer in attn_W_v],
    "attn_W_o": [layer.state_dict() for layer in attn_W_o],
    "ffn_W1": [w.detach().cpu() for w in ffn_W1],
    "ffn_b1": [b.detach().cpu() for b in ffn_b1],
    "ffn_W2": [w.detach().cpu() for w in ffn_W2],
    "ffn_b2": [b.detach().cpu() for b in ffn_b2],
}, "gpt_lm_sequential.pt")
print("Saved pretrained LM to gpt_lm_sequential.pt")

=== Pretraining LM ===


Epoch 1/10: 100%|██████████| 2/2 [00:00<00:00,  2.42it/s]


Epoch 1 LM loss: 3.5084


Epoch 2/10: 100%|██████████| 2/2 [00:00<00:00, 137.25it/s]


Epoch 2 LM loss: 3.1822


...

Epoch 8 LM loss: 2.0660


Epoch 9/10: 100%|██████████| 2/2 [00:00<00:00, 116.43it/s]


Epoch 9 LM loss: 1.9483


Epoch 10/10: 100%|██████████| 2/2 [00:00<00:00, 134.59it/s]


Epoch 10 LM loss: 1.8785

=== Demo: Causal LM next-token prediction ===
Prompt: 'this movie is' -> Generated: 'this movie is movie a is a is'
Prompt: 'the food was' -> Generated: 'the food was movie a is a is'
Prompt: 'today is a' -> Generated: 'today is a movie a is a is'
Prompt: 'i really' -> Generated: 'i really this movie a is a'
Saved pretrained LM to gpt_lm_sequential.pt


In [11]:
# ============================================================
# STEP 9: Add small classifier head (trainable)
# ============================================================
classifier = nn.Linear(d_model, 2).to(device)  # 2 labels: pos/neg
clf_optimizer = torch.optim.AdamW(classifier.parameters(), lr=1e-3)
print("=== Fine-tuning classifier on frozen GPT (few-shot sentiment) ===")
epochs_clf = 30
for epoch in range(epochs_clf):
    total_loss = 0.0
    correct = 0
    total = 0
    for inp, labels in tqdm(sent_loader, desc=f"Epoch {epoch+1}/{epochs_clf}"):
        inp = inp.to(device)
        labels = labels.to(device)
        clf_optimizer.zero_grad()
        # forward through frozen GPT (no grad)
        with torch.no_grad():
            _, _, hidden_states = gpt_forward_lm(inp, targets=None)  # [B, T, D]
        cls_rep = hidden_states[:, 0, :]  # <cls> position
        logits = classifier(cls_rep)      # [B, 2]
        loss = F.cross_entropy(logits, labels)
        loss.backward()
        clf_optimizer.step()
        total_loss += loss.item()
        preds = logits.argmax(dim=-1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    acc = correct / total if total > 0 else 0.0
    print(f"Epoch {epoch+1} clf loss: {total_loss/len(sent_loader):.4f}, acc: {acc:.2f}")


=== Fine-tuning classifier on frozen GPT (few-shot sentiment) ===


Epoch 1/30: 100%|██████████| 2/2 [00:00<00:00, 261.85it/s]


Epoch 1 clf loss: 0.8158, acc: 0.50


Epoch 2/30: 100%|██████████| 2/2 [00:00<00:00, 279.09it/s]


Epoch 2 clf loss: 0.7667, acc: 0.50


...

Epoch 28 clf loss: 0.6933, acc: 0.50


Epoch 29/30: 100%|██████████| 2/2 [00:00<00:00, 247.99it/s]


Epoch 29 clf loss: 0.6959, acc: 0.50


Epoch 30/30: 100%|██████████| 2/2 [00:00<00:00, 250.83it/s]

Epoch 30 clf loss: 0.6967, acc: 0.50


In [12]:
# ============================================================
# STEP 10: Test on new sentences
# ============================================================
classifier.eval()
test_sentences = [
    "i really loved this movie",
    "this film was horrible",
    "the food was very good",
    "the food was very bad",
]
with torch.no_grad():
    for text in test_sentences:
        ids = encode(text, add_cls=True)
        if len(ids) > max_len:
            ids = ids[:max_len]
        pad_len = max_len - len(ids)
        ids = ids + [pad_id] * pad_len
        ids_t = torch.tensor([ids], dtype=torch.long).to(device)
        _, _, hidden_states = gpt_forward_lm(ids_t, targets=None)
        cls_rep = hidden_states[:, 0, :]
        logits = classifier(cls_rep)
        pred = logits.argmax(dim=-1).item()
        label = "POSITIVE" if pred == 1 else "NEGATIVE"
        print(f"Text: {text} -> {label}")


Text: i really loved this movie -> POSITIVE
Text: this film was horrible -> POSITIVE
Text: the food was very good -> POSITIVE
Text: the food was very bad -> POSITIVE
